# Project 2 — Your Transformer, Now a Recommender

Treat each **movie** as a token and each user's **watch history** as a sentence:
next-token prediction becomes **next-movie prediction**. You train your
Project-1 transformer on MovieLens, pull out the learned item embeddings, build a
seed-movie similarity tool, study the embedding space with unsupervised methods,
and compare against classic baselines.

> **Use your own transformer.** Replace the files in `tinytransformer/` with your
> Project-1 versions. If yours had bugs, keep the reference files — you lose no
> Project-2 credit for a Project-1 bug. The graded code lives in
> `tinytransformer/rec/`.

In [ ]:
import matplotlib.pyplot as plt
import torch

from tinytransformer import TinyTransformerLM
from tinytransformer.rec import (
    data, build_training_sequences, train_seqrec,
    item_embeddings, similar_items, recommend_similar,
    run_kmeans, genre_purity, primary_genre_list,
    PopularityRecommender, ItemItemKNN, TransformerRecommender,
    build_eval_pairs, evaluate_ranking, viz,
)
torch.manual_seed(0)

## Get the data  *(provided)*

`load_movielens()` downloads and preprocesses **MovieLens ml-latest-small**. If
the download is unavailable (offline / proxy), we fall back to a synthetic
dataset with the same schema and planted genre + sequential structure.

In [ ]:
try:
    d = data.load_movielens(min_rating=4.0, min_user_len=5, min_item_count=5, max_len=50)
    print("Loaded real MovieLens ml-latest-small")
except Exception as e:
    print(f"Download unavailable ({type(e).__name__}); using synthetic dataset")
    d = data.make_synthetic_movielens(n_genres=6, items_per_genre=30,
                                      n_users=800, hist_len=24, seed=1, max_len=50)

print(f"items={d.n_items}  users={len(d.full_histories)}  genres={len(d.genres)}")
print("example history (first 8 ids):", d.full_histories[0][:8])

## Part A — histories → training tensors (`rec/dataset.py`)


Right-pad each user's history into aligned (input, target) next-item tensors. 

| Function | What you implement |
|----------|--------------------|
| `build_training_sequences` | right-pad each user's history into (input, target) next-item tensors |
| `get_batch` | sample a reproducible minibatch |

In [ ]:
X, Y = build_training_sequences(d.train_histories, d.max_len, d.pad_id)
print("X:", tuple(X.shape), " Y:", tuple(Y.shape))
print("row 0 input :", X[0][:12].tolist())
print("row 0 target:", Y[0][:12].tolist())   # shifted by one

## Part B — train the recommender (`rec/train.py`)

Same loop as Project 1, but the vocabulary is the item set and padded targets
are ignored in the loss.


| Function | What you implement |
|----------|--------------------|
| `train_seqrec` | the training loop; next-item cross-entropy with padded targets ignored (`ignore_index=pad_id`) |

In [ ]:
model = TinyTransformerLM(vocab_size=d.vocab_size, d_model=64, n_heads=4,
                         n_layers=2, block_size=d.max_len, dropout=0.1)
res = train_seqrec(model, X, Y, steps=1500, batch_size=64, lr=3e-3,
                   pad_id=d.pad_id, log_every=300)
plt.plot(res["losses"]); plt.xlabel("step"); plt.ylabel("loss")
plt.title(f"training loss  ({res['losses'][0]:.2f} -> {res['final_loss']:.2f})"); plt.show()

## Part C — pull out the item embeddings (`rec/embeddings.py`)
| Function | What you implement |
|----------|--------------------|
| `item_embeddings` | return the per-item vectors; choose the **input** (`tok_emb`) or **output** (`head`) embedding |

Your model holds **two** vectors per item: the input embedding (`tok_emb`, item
as context) and the output embedding (`head`, item as prediction target). You
will compare them below.

In [ ]:
emb_out = item_embeddings(model, which="output")
emb_in  = item_embeddings(model, which="input")
print("output emb:", tuple(emb_out.shape), " input emb:", tuple(emb_in.shape))

## Part D — the seed-program similarity tool (`rec/similarity.py`)
| Function | What you implement |
|----------|--------------------|
| `similar_items` | top-k cosine neighbours of a seed movie in embedding space |


Given a seed movie, return its nearest neighbours by cosine similarity in
embedding space. This is unsupervised retrieval — no genre labels are used.

In [ ]:
seed_query = "Toy Story"        # a title substring (real data) ...
try:
    seed_id, hits = recommend_similar(seed_query, d, emb_out, k=10)
except KeyError:
    seed_id, hits = recommend_similar(3, d, emb_out, k=10)   # ... or an id (synthetic)

print(f"Seed: {d.title(seed_id)}  [{d.item_primary_genre[seed_id]}]\n")
for title, genre, score in hits:
    print(f"  {score:.3f}  {title}  [{genre}]")

## Part E — unsupervised structure in the embedding space (`rec/cluster.py`)
| Function | What you implement |
|----------|--------------------|
| `run_kmeans` | k-means over the item embeddings |
| `genre_purity` | how well clusters line up with held-out genre labels |


You never trained on genre. Do the embeddings recover it anyway? Cluster them
and *measure* the overlap with held-out genre labels, then look at the t-SNE map.

In [ ]:
pg = primary_genre_list(d)
k = min(len(d.genres), 12)
pur_out = genre_purity(run_kmeans(emb_out, k, seed=0), pg)
pur_in  = genre_purity(run_kmeans(emb_in,  k, seed=0), pg)
print(f"k-means genre purity   output={pur_out:.3f}   input={pur_in:.3f}   (chance ~ {1/len(d.genres):.3f})")

In [ ]:
viz.plot_tsne(emb_out, d, top_genres=8, seed=0)
plt.show()

### Q1 — input vs. output embedding *(answer in markdown below)*

Which embedding recovers genre, and **why**? Think about what shapes each table
during training: the input embedding is what the model *reads*, the output
embedding (a row of the final linear layer) is what it *predicts*.

*Your answer here (a short paragraph).*

## Part F — baselines & comparison (`rec/baselines.py`, `rec/ranking_eval.py`)
| Function | What you implement |
|----------|--------------------|
| `PopularityRecommender` | most-popular ranking |
| `ItemItemKNN` | item-item cosine collaborative filtering |
| `evaluate_ranking` | leave-one-out hit@k / ndcg@k / MRR over all users |


Compare your transformer against a most-popular baseline and item-item cosine
kNN, using leave-one-out hit@10 / ndcg@10 / MRR.

In [ ]:
pairs = build_eval_pairs(d)
recs = {
    "popularity":    PopularityRecommender().fit(d.train_histories),
    "item-item kNN": ItemItemKNN().fit(d.train_histories),
    "transformer":   TransformerRecommender(model, pad_id=d.pad_id),
}
print(f"{'model':16s}{'hit@10':>9}{'ndcg@10':>10}{'mrr':>8}")
for name, rec in recs.items():
    r = evaluate_ranking(rec, pairs, k=10)
    print(f"{name:16s}{r['hit@k']:9.3f}{r['ndcg@k']:10.3f}{r['mrr']:8.3f}")

### Q2 — when does the transformer earn its keep? *(answer in markdown below)*

Explain the ordering in the table, and name one situation where the simplest
baseline is hard to beat.

*Your answer here (a short paragraph).*

## Caveat to keep in mind

MovieLens has **no plot text**, so these embeddings encode *co-watching
behaviour*, not content. "Similar to Inception" means "watched by the same
people," which usually — but not always — matches content. That gap is exactly
what a *text*-based recommender (the news project) would close.